# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [3]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [4]:
colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

corr = df[colunas_numericas].corr()

print("Matriz de Correlação")
display(corr)

fig = px.imshow(
    corr,
    text_auto=".3f",
    title="Matriz de Correlação"
)

fig.show()

display(df[colunas_numericas].describe())

Matriz de Correlação


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
taxa_abandono_carrinho_pct,1.000,-0.068,-0.116,-0.643
profundidade_scroll_pct,-0.068,1.000,0.050,0.485
tempo_primeiro_clique_s,-0.116,0.050,1.000,-0.229
taxa_conversao_pct,-0.643,0.485,-0.229,1.000


,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
count,180.000,180.000,180.000,180.000
mean,47.546,62.449,6.971,5.869
std,6.939,12.166,2.244,0.645
min,30.944,31.200,2.000,4.352
25%,42.697,54.099,5.477,5.434
50%,47.466,62.511,7.072,5.916
75%,51.916,69.722,8.466,6.307
max,71.311,95.000,12.715,7.753


In [5]:
variavel_x = "taxa_abandono_carrinho_pct"

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    trendline="ols",
    title="Taxa de abandono do carrinho vs Taxa de conversão",
)

fig.show()

In [6]:
fig = px.scatter(
    df,
    x="profundidade_scroll_pct",
    y="taxa_conversao_pct",
    trendline="ols",
    title="Profundidade de scroll vs Taxa de conversão",
)

fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Decidi priorizar a Taxa de Abandono de Carrinho e a Profundidade de Scroll. A taxa de abandono me chamou muita atenção por ter a maior correlação negativa absoluta do conjunto, ficando evidente como o principal gargalo das conversões. Já a profundidade de scroll me mostrou que usuários que engajam mais e exploram a interface convertem com mais frequência. Escolhi focar nessas duas porque, quando elas ficaram lado a lado com o tempo do primeiro clique, foram elas que me deram os sinais mais fortes sobre o comportamento real do usuário.

## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [7]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

,métrica,valor
0,MAE,0.276
1,RMSE,0.344


In [8]:
coef_df = pd.DataFrame({
    "Variável": ["Intercepto"] + features,
    "Coeficiente": coeficientes
})

display(coef_df)

,Variável,Coeficiente
0,Intercepto,7.883
1,taxa_abandono_carrinho_pct,-0.060
2,profundidade_scroll_pct,0.024
3,tempo_primeiro_clique_s,-0.094


In [9]:
ss_res = np.sum((y - pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)

r2 = 1 - (ss_res / ss_tot)

print(f"R² = {r2:.4f}")

R² = 0.7144


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

Notei que o erro médio absoluto ficou bem baixo quando comparado com a escala da taxa de conversão, o que me indica que o modelo conseguiu reproduzir muito bem o comportamento dos dados. Além disso, vi que o RMSE ficou só um pouco acima do MAE, o que sugere que não temos erros extremos que possam comprometer a qualidade das previsões e o valor do R² me confirmou que boa parte da variação que vemos na taxa de conversão é realmente explicada pelas variáveis que utilizei.

## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [10]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [11]:
variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct"
]

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:

    linha_cenario = linha_base.copy()

    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)

    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)

    variacao_saida = (saida_nova - saida_base) / saida_base

    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)

display(tabela_sensibilidade)

,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258


In [12]:
fig = px.bar(
    tabela_sensibilidade,
    x="variável",
    y="índice_sensibilidade",
    title="Comparação dos Índices de Sensibilidade"
)

fig.show()

Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

In [13]:
mais_importante = tabela_sensibilidade.iloc[
    tabela_sensibilidade["índice_sensibilidade"].abs().idxmax()
]

print(f"""
A variável com maior impacto foi:

{mais_importante['variável']}

Seu índice de sensibilidade foi de
{mais_importante['índice_sensibilidade']:.3f}.

Isso significa que uma alteração de 10% nessa variável produz
uma mudança proporcionalmente maior na taxa de conversão prevista.

Portanto, essa variável deve receber prioridade nas decisões de produto.
""")


A variável com maior impacto foi:

taxa_abandono_carrinho_pct

Seu índice de sensibilidade foi de
-0.489.

Isso significa que uma alteração de 10% nessa variável produz
uma mudança proporcionalmente maior na taxa de conversão prevista.

Portanto, essa variável deve receber prioridade nas decisões de produto.



## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

A minha principal recomendação é priorizar as ações que reduzem a taxa de abandono do carrinho, porque essa variável foi a que apresentou o maior impacto na análise de sensibilidade, o que indica que pequenas melhorias nela já geram mudanças bem relevantes na nossa taxa de conversão prevista. Para resolver isso, eu sugiro focar em ações como simplificar o checkout, reduzir as etapas da compra, exibir os custos finais mais cedo e melhorar o desempenho da página.

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

Sobre as limitações e riscos, é importante pontuar que a análise que eu fiz assume relações lineares entre as variáveis e a taxa de conversão, o que pode não se manter indefinidamente na prática. Além disso, os dados que foram utilizados são simulados, então eles não representam necessariamente o comportamento 100% real dos usuários. Por isso, acredito que a gente deve interpretar esses resultados muito mais como um apoio direcional para guiar a nossa decisão do que como uma previsão exata de como as coisas vão acontecer no mundo real.

In [18]:
# Use este espaço para responder a questão. Use quantas células julgar necessário

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [17]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

,0
count,1000.000
mean,5.869
std,0.393
min,4.654
10%,5.369
25%,5.617
50%,5.854
75%,6.131
90%,6.379
max,7.193


In [19]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

In [20]:
p5 = np.percentile(previsoes, 5)
p95 = np.percentile(previsoes, 95)

print(f"Percentil 5% : {p5:.3f}")
print(f"Percentil 95%: {p95:.3f}")

Percentil 5% : 5.221
Percentil 95%: 6.539


In [21]:
prob = np.mean(previsoes > 5)

print(
    f"Probabilidade da conversão prevista ser maior que 5%: {prob:.2%}"
)

Probabilidade da conversão prevista ser maior que 5%: 99.00%


In [22]:
importancia = []

for i, var in enumerate(features):

    beta_padronizado = (
        coeficientes[i+1]
        * df[var].std()
        / df[target].std()
    )

    importancia.append({
        "Variável": var,
        "Beta Padronizado": beta_padronizado
    })

importancia_df = pd.DataFrame(importancia)

display(
    importancia_df.sort_values(
        by="Beta Padronizado",
        key=np.abs,
        ascending=False
    )
)

,Variável,Beta Padronizado
0,taxa_abandono_carrinho_pct,-0.650
1,profundidade_scroll_pct,0.457
2,tempo_primeiro_clique_s,-0.328


A simulação de Monte Carlo mostrou que a taxa de conversão fica bem concentrada em torno da média que o modelo previu. A maior parte dos cenários simulados ficaram entre 5.2% e 6.5% de conversão, o que sugere que o sistema possui um comportamento relativamente estável mesmo com incerteza nas variáveis de entrada. Por isso, a minha recomendação de focar em reduzir a taxa de abandono continua válida, já que essa variável apresentou uma sensibilidade maior e mantém a sua importância mesmo diante dessa variabilidade que simulei. No fim das contas, eu considero o risco dessa recomendação de baixo a moderado, já que a distribuição das previsões não apresentou uma dispersão excessiva.

## Bônus Executivo: Qual o impacto financeiro dessa decisão?

Para ir um pouco além da estatística, resolvi traduzir o impacto da minha recomendação de reduzir o abandono de carrinho para a linguagem de negócios.

Criei uma simulação financeira rápida assumindo premissas fictícias, mas realistas para um e-commerce de tráfego de 10.000 usuários diários e ticket médio de R$ 150,00 por compra.

A ideia aqui é responder se a equipe de engenharia conseguir reduzir a taxa de abandono em 10%, quanto isso coloca de dinheiro na mesa no final do mês?

In [23]:
import plotly.graph_objects as go

# Premissas de negócio
usuarios_diarios = 10000
ticket_medio = 150.00
dias_no_mes = 30

# Calculando a conversão atual usando a linha base do modelo
conversao_atual_pct = prever_linha(linha_base)

# Simulando o cenário com 10% a menos de abandono de carrinho
linha_otimizada = linha_base.copy()
linha_otimizada["taxa_abandono_carrinho_pct"] = linha_base["taxa_abandono_carrinho_pct"] * 0.90
conversao_otimizada_pct = prever_linha(linha_otimizada)

# Traduzindo conversão em receita mensal
vendas_atuais = usuarios_diarios * (conversao_atual_pct / 100) * dias_no_mes
receita_atual = vendas_atuais * ticket_medio

vendas_otimizadas = usuarios_diarios * (conversao_otimizada_pct / 100) * dias_no_mes
receita_otimizada = vendas_otimizadas * ticket_medio

# O "Uplift" é o ganho financeiro direto da nossa ação de UX
uplift_receita = receita_otimizada - receita_atual

# Criando um Gráfico Waterfall para apresentar o ganho
fig_fin = go.Figure(go.Waterfall(
    name="Impacto UX",
    orientation="v",
    measure=["absolute", "relative", "total"],
    x=["Receita Atual (Mês)", "Uplift (Melhoria no Checkout)", "Receita Projetada (Mês)"],
    textposition="outside",
    text=[
        f"R$ {receita_atual/1e6:.2f}M",
        f"+ R$ {uplift_receita/1e3:.0f} mil",
        f"R$ {receita_otimizada/1e6:.2f}M"
    ],
    y=[receita_atual, uplift_receita, receita_otimizada],
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    increasing={"marker": {"color": "#2ca02c"}},
    totals={"marker": {"color": "#1f77b4"}}
))

fig_fin.update_layout(
    title="Projeção de Impacto Financeiro (Aumento de Receita Mensal)",
    showlegend=False,
    plot_bgcolor="white"
)

fig_fin.show()

Com o gráfico acima, a defesa da minha análise fica muito mais palpável. É possível notar que tem uma melhoria de apenas 10% na métrica de abandono de carrinho, então acaba não não sendo só um "ganho de interface", mas também ela tem o potencial de gerar dezenas de milhares de reais em receita adicional (Uplift) todos os meses sem precisarmos gastar a mais trazendo novos usuários para o aplicativo. É por isso que otimizar esse gargalo é a escolha mais inteligente para o próximo ciclo do produto.

## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.